<a href="https://colab.research.google.com/github/williamryanmckee-nwa/Deep-Learning/blob/main/12_05_Introduction_to_BERT__V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# __Introduction to BERT and Transformers Library__
- BERT stands for Bidirectional Encoder Representations from Transformers.
- BERT is pre-trained on a large corpus of unlabeled text, including the entire Wikipedia (that's 2,500 million words!) and the Book Corpus (800 million words).
- BERT is based on the Transformer architecture.

## Steps to be followed:
1. Import the required libraries
2. Analyze the sentiment using the transformer pipeline
3. Create text generation
4. Create named entity recognition (NER)
5. Generate a masked language model using a model and a tokenizer

### Step 1: Import the required libraries
- The code from the transformers import pipeline allows for easy access to pre-trained models and simplified execution of NLP tasks using the transformers library.



In [13]:
!pip install --upgrade transformers huggingface_hub

  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)


In [14]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu125

Looking in indexes: https://download.pytorch.org/whl/cu125


In [ ]:
pip install tf-keras

In [ ]:
import os

# Disable oneDNN optimizations to avoid potential minor numerical differences caused by floating-point round-off errors.
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# Set CUDA_VISIBLE_DEVICES:
#   "-1" -> force CPU-only execution
#   "0"  -> use first GPU (change to "1", "2", etc. for other GPUs)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


In [ ]:
import torchvision

torchvision.disable_beta_transforms_warning()

from transformers import pipeline

###Step 2: Analyze the sentiment using  transformer pipeline

- Import the pipeline function from the Transformers library, which enables easy access to pre-trained NLP models.
- The snippet creates a sentiment analysis pipeline using the pre-trained model and uses it to classify the sentiment of the input text, **I hate you.**
- The result, including the sentiment label and score, is then printed.

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

result = classifier("I hate you")[0]
print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

- Perform sentiment analysis on the text **I love you.**
- Print the sentiment analysis result.

In [ ]:
result = classifier("I love you")[0]
print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

**Observation**
- The sentiment analysis model is highly confident that the sentiment of the text **I love you** is positive, with a score of 0.9999.

### Step 3: Create text generation
- It creates a text-generation pipeline using the pipeline function from the Transformers library.
- It generates text starting with the provided prompt **As far as I am concerned, I will** using the text generation pipeline with a maximum length of 50 tokens and without sampling, which is deterministic output.
- The generated text is then printed.

In [ ]:
text_generator = pipeline("text-generation", model="gpt2", max_length=50)
print(text_generator("As far as I am concerned, I will", max_length=20, do_sample=False))


### Step 4: Create named entity recognition (NER)
- It creates a NER pipeline using the pipeline function from the Transformers library.

- It applies the NER pipeline to the provided sequence, which is a text containing named entities. The pipeline identifies and extracts named entities such as organization names **Hugging Face Inc.**, locations **New York City**, and others. The extracted entities are then printed.

In [ ]:
ner_pipe = pipeline("ner")
sequence = """Hugging Face Inc. is a company based in New york city. Manhattan bridge is visible from the window."""

- Print the entities after performing named entity recognition on the sequence.

In [ ]:
for entity in ner_pipe(sequence):
    print(entity)


### Step 5: Generate a masked language model using a model and a tokenizer

- Masked language modeling using a model and a tokenizer
  - Masked language modeling is a task where a model fills in masked tokens in a sequence, improving its understanding of language. It involves predicting missing tokens by considering the context of surrounding words.

- The process includes the following steps:
  - Instantiate a tokenizer and a model from the checkpoint name.
  - Define a sequence with a masked token, placing the tokenizer.mask_token instead of a word.
  - Encode that sequence into a list of IDs and find the position of the masked token in that list.
  - Retrieve the predictions at the index of the masked token.
  - Retrieve the top 5 tokens using the PyTorch topk or TensorFlow top_k methods.
  - Replace the masked token with the tokens and print the results.

 ### Masked language modeling
- Import the necessary modules from the transformer library and torch.
- Load the pre-trained tokenizer and model.
- Define the input sequence with a masked token.
- Tokenize the input sequence and convert it to tensors.
- Find the index of the masked token and generate token predictions using the model.
- Get the indices of the top 5 predicted tokens and print them in the sequence.
- Load the pre-trained tokenizer and model.
- Define the input sequence with a masked token.
- Tokenize the input sequence and convert it to tensors.
- Find the index of the masked token and generate token predictions using the model.
- Get the indices of the top 5 predicted tokens and print them in the sequence.
- Print the top 5 predicted tokens in the masked position.

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")
model = AutoModelForMaskedLM.from_pretrained("distilbert-base-cased")

sequence = (
    "Distilled models are smaller than the models they mimic. Using them instead of the large "
    f"versions would help {tokenizer.mask_token} our carbon footprint."
)

inputs = tokenizer(sequence, return_tensors="pt")
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

token_logits = model(**inputs).logits
mask_token_logits = token_logits[0, mask_token_index, :]

top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(sequence.replace(tokenizer.mask_token, tokenizer.decode([token])))

**Observation**
- The output provides alternative sentence suggestions by replacing the masked token with different predicted tokens, demonstrating how using distilled models instead of larger ones can impact the carbon footprint.